In [1]:
# API 키를 환경변수로 관리하기 위한 설정 파일
from dotenv import load_dotenv

# API 키 정보 로드
load_dotenv()

True

In [2]:
import faiss
from langchain_community.vectorstores import FAISS
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from langchain_core.prompts import PromptTemplate
from langchain_community.chat_models import ChatOllama
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_teddynote.messages import stream_response

In [3]:
model_name = "intfloat/multilingual-e5-large-instruct"
hf_embeddings = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs={"device": "cuda"},
    encode_kwargs={"normalize_embeddings": True},
)

In [4]:
# 저장된 데이터를 로드
db = FAISS.load_local(
    folder_path="faiss_db",
    index_name="only266927",
    embeddings=hf_embeddings,
    allow_dangerous_deserialization=True,
)

In [5]:
retriever = db.as_retriever(search_type="mmr", search_kwargs={"k": 5, "lambda_mult": 0.25, "fetch_k": 10})
retriever.invoke("소득이 10만원인 사람은 어떻게 세금이 매겨질까?")

[Document(metadata={'law': '조세특례제한법 시행규칙', 'date': datetime.date(2024, 12, 3), 'chapter': '', 'passage': ''}, page_content='10의2. 영 제23조제17항에 따른 중간예납세액신고서: 별지 제9호의2서식\n10의3. 영 제24조제14항, 영 제24조의2제6항, 영 제81조의4제6항, 영 제82조의2제6항, 영 제83조제12항, 영 제92조의13제10항 및 영 제93조의3제9항에 따른 금융투자소득 합산과세 신청서: 별지 제9호의3서식\n10의4. 영 제25조제9항, 영 제25조의2제9항, 영 제25조의3제10항 및 영 제25조의4제9항ㆍ제10항에 따른 감가상각비조정명세서 및 감가상각비조정명세서합계표: 별지 제9호의4서식(1), 별지 제9호의4서식(2) 및 별지 제9호의4서식(3)\n10의5. 영 제25조제9항, 영 제25조의2제9항, 영 제25조의3제10항 및 영 제25조의4제4항에 따른 내용연수 특례적용 신청서: 별지 제9호의5서식\n11. 영 제26조의2제3항에 따른 산업수요맞춤형고등학교등 졸업자 복직 중소기업 세액공제신청서: 별지 제10호서식\n11의2. 영 제26조의3제6항에 따른 세액공제신청서: 별지 제10호의2서식(1), 별지 제10호의2서식(2)\n11의3. 영 제26조의4제17항에 따른 세액공제신청서: 별지 제10호의3서식 또는 별지 제10호의4서식\n11의4. 영 제26조의5제11항에 따른 공제세액계산서: 별지 제10호의5서식\n11의5. 영 제26조의6제4항에 따른 감면신청서: 별지 제10호의6서식\n11의6. 영 제26조의6제5항에 따른 감면 대상 명세서: 별지 제10호의7서식\n11의7. 영 제26조의7제10항에 따른 공제세액계산서: 별지 제10호의8서식\n11의8. 영 제26조의8제11항에 따른 공제세액계산서: 별지 제10호의9서식\n12. 영 제27조제5항에 따른 감면신청서: 별지 제11호서식\n12의2. 영 제27조제6항에 따른 감면 대상 명

In [6]:
# 단계 6: 프롬프트 생성(Create Prompt)
# 프롬프트를 생성합니다.
prompt = PromptTemplate.from_template(
"""당신은 한국의 세금 관련 법을 바탕으로 답변을 수행하는 AI 어시스턴트입니다. 당신의 임무는 주어진 문맥(context) 에서 주어진 질문(question) 에 답하는 것입니다.
검색된 다음 문맥(context)를 사용하여 질문(question)에 답하세요. 만약, 주어진 문맥(context)에서 답을 찾을 수 없다면, 답을 모른다면 "주어진 질문에 대한 정보를 찾을 수 없습니다."라고 답하세요.
한글로 답변하고, 답변에 참고한 문맥(context)를 답변 하단에 작성해주세요.

#Question: 
{question} 
#Context: 
{context} 

#Answer:"""
)

In [7]:
# Ollama 모델을 불러옵니다.
llm = ChatOllama(model="llama3.2")

# 체인을 생성합니다.
rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

C:\Users\kangs\AppData\Local\Temp\ipykernel_3052\4059665236.py:2: LangChainDeprecationWarning: The class `ChatOllama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import ChatOllama``.
  llm = ChatOllama(model="llama3.2")


In [8]:
answer = rag_chain.stream("소득이 10만원인 사람은 어떻게 세금이 매겨질까?")
stream_response(answer)

주민가eld는 소득세법 제20조제2항에 따라 정해진 총급여액입니다.

우선, 이와 같은 소득은 영수증이나 신용카드 매출 전표 등에서 확인하여 수입금액으로 계산합니다. 

그 후, 이 금액을 연체일수에 따라 산정한 월평균 근로소득으로 계산합니다.

In [12]:
rag_chain.get_graph().print_ascii()

           +---------------------------------+         
           | Parallel<context,question>Input |         
           +---------------------------------+         
                    **               **                
                 ***                   ***             
               **                         **           
+----------------------+              +-------------+  
| VectorStoreRetriever |              | Passthrough |  
+----------------------+              +-------------+  
                    **               **                
                      ***         ***                  
                         **     **                     
           +----------------------------------+        
           | Parallel<context,question>Output |        
           +----------------------------------+        
                             *                         
                             *                         
                             *                  

In [13]:
rag_chain.get_prompts()

[PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='당신은 한국의 세금 관련 법을 바탕으로 답변을 수행하는 AI 어시스턴트입니다. 당신의 임무는 주어진 문맥(context) 에서 주어진 질문(question) 에 답하는 것입니다.\n검색된 다음 문맥(context)를 사용하여 질문(question)에 답하세요. 만약, 주어진 문맥(context)에서 답을 찾을 수 없다면, 답을 모른다면 "주어진 질문에 대한 정보를 찾을 수 없습니다."라고 답하세요.\n한글로 답변하고, 전문 용어 및 고유 명사 등 모르는 단어는 검색된 문장 그대로 보여주셔도 좋습니다.\n\n#Question: \n{question} \n#Context: \n{context} \n\n#Answer:')]